In [ ]:
import copy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------------------------
# 데이터 전처리
# ---------------------------
train_df = pd.read_csv('data_preprocess/train_e.csv')
test_df = pd.read_csv('data_preprocess/test_e.csv')

X = train_df.drop(["임신 성공 여부"], axis=1)
y = train_df["임신 성공 여부"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(test_df)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).to(device)
X_val_tensor   = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor   = torch.tensor(y_val.to_numpy(), dtype=torch.float32).to(device)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# WeightedRandomSampler 적용
class_sample_count = np.array([(y_train == t).sum() for t in np.unique(y_train)])
weights = 1. / class_sample_count
samples_weight = np.array([weights[int(t)] for t in y_train])
samples_weight = torch.from_numpy(samples_weight).float()
sampler = WeightedRandomSampler(samples_weight, len(samples_weight), replacement=True)
train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler)

# ---------------------------
# Focal Loss 정의
# ---------------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        BCE_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        focal_loss = (1 - pt) ** self.gamma * BCE_loss
        # alpha 적용: 0.25는 소수 클래스에 집중하기 위한 값 (데이터에 따라 조정)
        focal_loss = self.alpha * focal_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

criterion = FocalLoss(gamma=2.0, alpha=0.25)

# ---------------------------
# 모델 정의 (Residual 연결 포함, 모델 크기: 512-256-128)
# ---------------------------
class SelfAttention(nn.Module):
    def __init__(self, input_dim):
        super(SelfAttention, self).__init__()
        self.query = nn.Linear(input_dim, input_dim)
        self.key   = nn.Linear(input_dim, input_dim)
        self.value = nn.Linear(input_dim, input_dim)
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x):
        residual = x
        x_unsq = x.unsqueeze(1)
        Q = self.query(x_unsq)
        K = self.key(x_unsq)
        V = self.value(x_unsq)
        attention_scores = self.softmax(torch.bmm(Q, K.transpose(1, 2)) / (x.shape[-1] ** 0.5))
        attn_output = torch.bmm(attention_scores, V).squeeze(1)
        return residual + attn_output

class FeatureAttention(nn.Module):
    def __init__(self, input_dim):
        super(FeatureAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(),
            nn.Softmax(dim=1)
        )
    
    def forward(self, x):
        attn_weights = self.attention(x)
        out = x * attn_weights
        return x + out

class MLPModel(nn.Module):
    def __init__(self, input_dim):
        super(MLPModel, self).__init__()
        self.feature_attention = FeatureAttention(input_dim)
        self.self_attention    = SelfAttention(input_dim)
        
        self.fc1 = nn.Linear(input_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(0.3)
        
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(0.3)
        
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.dropout3 = nn.Dropout(0.3)
        
        self.fc4 = nn.Linear(128, 1)  # logits 출력
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.feature_attention(x)
        x = self.self_attention(x)
        
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout3(x)
        
        x = self.fc4(x)
        return x

model = MLPModel(X_train.shape[1]).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# ---------------------------
# 학습 루프 (워밍업 스케줄러 예시 포함)
# ---------------------------
best_valid_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
patience = 5
counter = 0
epochs = 15
warmup_epochs = 3

for epoch in range(epochs):
    # 워밍업 적용
    if epoch < warmup_epochs:
        lr_scale = (epoch + 1) / warmup_epochs
        for param_group in optimizer.param_groups:
            param_group['lr'] = 0.001 * lr_scale
    else:
        scheduler.step()

    model.train()
    total_train_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_tensor).squeeze()
        valid_loss = criterion(val_logits, y_val_tensor)
        y_val_proba = torch.sigmoid(val_logits).cpu().numpy()
        y_val_pred = (y_val_proba > 0.5).astype(int)
    
    auc_pr = average_precision_score(y_val, y_val_proba)
    roc_auc = roc_auc_score(y_val, y_val_proba)
    
    print(f"Epoch {epoch+1}, Train Loss: {total_train_loss/len(train_loader):.4f}, "
          f"Valid Loss: {valid_loss.item():.4f}, AUC PR: {auc_pr:.4f}, ROC AUC: {roc_auc:.4f}")
    
    if valid_loss.item() < best_valid_loss:
        best_valid_loss = valid_loss.item()
        best_model_wts = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early Stopping triggered!")
            break

model.load_state_dict(best_model_wts)
model.eval()
with torch.no_grad():
    val_logits = model(X_val_tensor).squeeze()
    y_val_proba = torch.sigmoid(val_logits).cpu().numpy()
    y_val_pred = (y_val_proba > 0.5).astype(int)

print("Final Model Performance:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_proba))


Epoch 1, Train Loss: 0.0381, Valid Loss: 0.0369, AUC PR: 0.4403, ROC AUC: 0.7336
Epoch 2, Train Loss: 0.0370, Valid Loss: 0.0366, AUC PR: 0.4417, ROC AUC: 0.7356
Epoch 3, Train Loss: 0.0369, Valid Loss: 0.0375, AUC PR: 0.4455, ROC AUC: 0.7369
Epoch 4, Train Loss: 0.0368, Valid Loss: 0.0368, AUC PR: 0.4446, ROC AUC: 0.7381
Epoch 5, Train Loss: 0.0366, Valid Loss: 0.0376, AUC PR: 0.4468, ROC AUC: 0.7371
Epoch 6, Train Loss: 0.0365, Valid Loss: 0.0365, AUC PR: 0.4487, ROC AUC: 0.7390
Epoch 7, Train Loss: 0.0364, Valid Loss: 0.0372, AUC PR: 0.4482, ROC AUC: 0.7393
Epoch 8, Train Loss: 0.0364, Valid Loss: 0.0372, AUC PR: 0.4502, ROC AUC: 0.7396
Epoch 9, Train Loss: 0.0363, Valid Loss: 0.0374, AUC PR: 0.4505, ROC AUC: 0.7404
Epoch 10, Train Loss: 0.0363, Valid Loss: 0.0369, AUC PR: 0.4512, ROC AUC: 0.7409
Epoch 11, Train Loss: 0.0364, Valid Loss: 0.0367, AUC PR: 0.4512, ROC AUC: 0.7412
Early Stopping triggered!
Final Model Performance:
Accuracy: 0.608664926090968
              precision    r

In [11]:
# 전체 데이터로 재학습
model_full = MLPModel(X.shape[1]).to(device)
model_full.load_state_dict(best_model_wts)
model_full.eval()
with torch.no_grad():
    y_pred_proba = model_full(X_test_tensor).cpu().numpy().squeeze()

# 제출 파일 생성
sample_submission = pd.read_csv('simibe/Data/sample_submission.csv')
sample_submission['probability'] = y_pred_proba
sample_submission.to_csv('./mlp_submission.csv', index=False)
